In [ ]:
# 1. transformer 

In [5]:
import torch
from torch import nn
import math
import torch.nn.functional as F

In [9]:
# 构成词嵌入的参数
src_vocab_size = 70000   #词袋数量
target_vocab_size = 8000 #目标词袋的数量
d_model= 512             #词向量的维度

# 文本的长度
max_len = 5000      #序列的最大长度 【千问 把这个限制突破到十万以上】
seq_len = 36        #每个输入序列的长度（循环神经网络的循环次数）
target_seq_len = 64 #每个目标输入序列的长度

#头数
num_heads = 8       #注意：头数一定被d_model整除（就是 把输入的词向量均分为8个向量

#前馈神经网络的隐藏维数
d_ff = 2048         # 也可以1024


#编码器 = 编码器 * 层数
num_layers =6      

#批次大小
batch_size = 64

#drop_out 丢弃率
drop_out = 0.3

In [ ]:
# 词嵌入
encoder_embedding = nn.Embedding(src_vocab_size,d_model)        #利用了 torch的实现
# 【batch_size, sql_len, d_model】


In [11]:
# 模拟token【batch_size,seq_len】
src = torch.randint(0,10000,(batch_size,seq_len),dtype=torch.long)


encoder_embedding(src).shape

torch.Size([64, 36, 512])

In [12]:
#位置编码

- $\rm{PE}_{(pos, 2i)} = \sin ( \dfrac{\rm{pos}}{10000 ^ {2i/d\_model}} )$

- $\rm{PE}_{(pos,2i+1)} = \cos(\dfrac{pos}{10000^{2i/d\_model}})$

- 位置编码分母部分计算
    - 10000 ^ (-i / d_model)
    - (1/10000) ^ (i / d_model)
    - (exp( -ln(10000) / d_model )) ^ i
    - exp(i * (-ln(10000) / d_model))

In [13]:
import torch
#生成位置
max_len=5000
position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # [max_len, 1]
position.shape

torch.Size([5000, 1])

In [14]:
import math
# 位置编码的分母
i2 = torch.arange(0, d_model, 2).float() 
log = -math.log(10000.0) / d_model
div_term = torch.exp(i2 * log)
print(div_term)

tensor([1.0000e+00, 9.6466e-01, 9.3057e-01, 8.9769e-01, 8.6596e-01, 8.3536e-01,
        8.0584e-01, 7.7737e-01, 7.4989e-01, 7.2339e-01, 6.9783e-01, 6.7317e-01,
        6.4938e-01, 6.2643e-01, 6.0430e-01, 5.8294e-01, 5.6234e-01, 5.4247e-01,
        5.2330e-01, 5.0481e-01, 4.8697e-01, 4.6976e-01, 4.5316e-01, 4.3714e-01,
        4.2170e-01, 4.0679e-01, 3.9242e-01, 3.7855e-01, 3.6517e-01, 3.5227e-01,
        3.3982e-01, 3.2781e-01, 3.1623e-01, 3.0505e-01, 2.9427e-01, 2.8387e-01,
        2.7384e-01, 2.6416e-01, 2.5483e-01, 2.4582e-01, 2.3714e-01, 2.2876e-01,
        2.2067e-01, 2.1288e-01, 2.0535e-01, 1.9810e-01, 1.9110e-01, 1.8434e-01,
        1.7783e-01, 1.7154e-01, 1.6548e-01, 1.5963e-01, 1.5399e-01, 1.4855e-01,
        1.4330e-01, 1.3824e-01, 1.3335e-01, 1.2864e-01, 1.2409e-01, 1.1971e-01,
        1.1548e-01, 1.1140e-01, 1.0746e-01, 1.0366e-01, 1.0000e-01, 9.6466e-02,
        9.3057e-02, 8.9769e-02, 8.6596e-02, 8.3536e-02, 8.0584e-02, 7.7736e-02,
        7.4989e-02, 7.2339e-02, 6.9783e-

In [15]:
pe = torch.zeros(max_len, d_model)
pos = position * div_term
print(pos.shape)

torch.Size([5000, 256])


In [16]:
pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)

In [17]:
import torch
import torch.nn as nn
import math
class PositionalEncoding(nn.Module):
    """位置编码"""
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # 创建位置编码矩阵 [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # [max_len, 1]
        
        # 计算分母项
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # 应用sin和cos函数
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # 添加batch维度
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        
        # 注册为缓冲区（不参与训练）
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, d_model]
        Returns:
            [batch_size, seq_len, d_model]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [18]:
# 位置编码计算

In [19]:
position_encoding = PositionalEncoding(d_model,max_len)




In [20]:
# 
num_heads = 8
d_k = d_model // num_heads
W_q = nn.Linear(d_model, d_model)
W_k = nn.Linear(d_model, d_model)
W_v = nn.Linear(d_model, d_model)

# embed 带位置编码的词嵌入
Q = W_q(embed)  # [batch_size, query_len, d_model]
K = W_k(embed)    # [batch_size, key_len, d_model]
V = W_v(embed)  # [batch_size, value_len, d_model]
print(Q.shape, K.shape, V.shape)

NameError: name 'embed' is not defined

In [ ]:
Q = Q.view(batch_size, -1, num_heads, d_k).transpose(1, 2)  # [batch_szie, num_heads, seq_len, d_model/num_heads]
K = K.view(batch_size, -1, num_heads, d_k).transpose(1, 2)
V = V.view(batch_size, -1, num_heads, d_k).transpose(1, 2)
print(Q.shape, K.shape, V.shape)

- 缩放点积注意力计算
     - $\rm{Attention}(Q,K,V) = \text{softmax}\left( \dfrac{QK^T}{\sqrt{d_k}} \right) V$

In [ ]:
import torch.nn.functional as F
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    计算缩放点积注意力
    Args:
        Q: [batch_size, num_heads, seq_len, d_k]
        K: [batch_size, num_heads, seq_len, d_k]
        V: [batch_size, num_heads, seq_len, d_k]
        mask: [batch_size, seq_len, seq_len] 或 [batch_size, 1, seq_len, seq_len]
    Returns:
        output: [batch_size, num_heads, seq_len, d_k]
        attention_weights: [batch_size, num_heads, seq_len, seq_len]
    """
    # 计算注意力分数
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k) # 后面两个维度做转置。
    
    # 应用mask（如果需要）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)  # 掩码的位置填充-1e9表示无穷小（极小值）
    
    # 应用softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # 计算输出
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

In [ ]:
attn_output, attention_weights = scaled_dot_product_attention(Q, K, V, mask=None)
print(attn_output.shape, attention_weights.shape)